In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
spark.sql(F'USE CATALOG {catalog_name}')


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

PRODUCTS_SCHEMA = StructType(
    [
        StructField("product_id", IntegerType()),
        StructField("product_name",StringType()),
        StructField("category", StringType()),
        StructField("subcategory", StringType()),
        StructField("brand", StringType()),
        StructField("price", DecimalType()),
        StructField("cost", DecimalType()),
        StructField("supplier_id", StringType()),
        StructField("product_status", StringType()),
        StructField("updated_at", TimestampType()),
    ]
)

INVENTORY_SCHEMA = StructType(
    [
        StructField("inventory_date", DateType()),
        StructField("product_id", IntegerType()),
        StructField("warehouse_id", StringType()),
        StructField("available_quantity", IntegerType()),
        StructField("reserved_quantity", IntegerType()),
        StructField("reorder_level", IntegerType()),
    ]
)

CLICKSTREAM_SCHEMA = StructType(
    [
        StructField("event_id", LongType()),
        StructField("customer_id", DoubleType()),
        StructField("session_id", StringType()),
        StructField("event_time", TimestampType()),
        StructField("event_type", StringType()),
        StructField("product_id", IntegerType()),
        StructField("page", StringType()),
        StructField("device_type", StringType()),
    ]
)

In [0]:
products_df = (spark.readStream
    .format("csv")
    .option("header", "true")
    .schema(PRODUCTS_SCHEMA)
    .load("/Volumes/shopsphere/ecomm_schema/capstone_data/adls/initial/products/"))
 
products_df.writeStream.option(
    "checkpointLocation",
    "/Volumes/shopsphere/ecomm_schema/capstone_data/checkpoint/products",
).format('delta').trigger(availableNow=True).outputMode("Append").toTable(
    "shopsphere.bronze.products"
)

In [0]:
inventory_df = (spark.readStream
    .format("csv")
    .option("header", "true")
    .schema(INVENTORY_SCHEMA)
    .load("/Volumes/shopsphere/ecomm_schema/capstone_data/adls/initial/inventory/"))
 
inventory_df.writeStream.option(
    "checkpointLocation",
    "/Volumes/shopsphere/ecomm_schema/capstone_data/checkpoint/inventory",
).format('delta').trigger(availableNow=True).outputMode("Append").toTable(
    "shopsphere.bronze.inventory"
)

In [0]:
clickstream_df = (spark.readStream
    .format("json")
    .schema(CLICKSTREAM_SCHEMA)
    .load("/Volumes/shopsphere/ecomm_schema/capstone_data/adls/initial/clickstream/"))

clickstream_df = clickstream_df.withColumn("customer_id", col("customer_id").cast("integer"))
 
clickstream_df.writeStream.option(
    "checkpointLocation",
    "/Volumes/shopsphere/ecomm_schema/capstone_data/checkpoint/clickstream",
).format('delta').trigger(availableNow=True).outputMode("Append").toTable(
    "shopsphere.bronze.clickstream"
)